# Package

In [1]:
# ============================================================
# 1) Core Python & Paths
# ============================================================
import sys
from pathlib import Path
from tempfile import TemporaryDirectory
from dateutil.relativedelta import relativedelta

# Project root (notebook dans /notebooks)
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# ============================================================
# 2) Data & Utils (Feast + custom utils)
# ============================================================
import pandas as pd
import numpy as np

from utils import load_wide_from_feast, build_unrate_exog_dataset
from experiment_utils import to_wide_from_oos, make_mae_dm_pivot

# ============================================================
# 3) Forecasting (MLForecast + Models)
# ============================================================
from mlforecast import MLForecast
from mlforecast.utils import PredictionIntervals
from sklearn.linear_model import Ridge
from lightgbm import LGBMRegressor
from sklearn.model_selection import GridSearchCV

# ============================================================
# 4) Evaluation & Statistical Tests
# ============================================================
from scipy.stats import ttest_rel
from statsmodels.stats.contingency_tables import mcnemar

# ============================================================
# 5) Visualization
# ============================================================
from utilsforecast.plotting import plot_series
from IPython.display import IFrame, display

# ============================================================
# 6) Experiment Tracking (MLflow)
# ============================================================
import mlflow
from mlflow.tracking import MlflowClient
from sklearn.linear_model import LinearRegression

# Importation des données

In [2]:
series_ids = [
    "BUSLOANS","CPIAUCSL","DPCERA3M086SBEA","INDPRO",
    "M2SL","OILPRICEX","RPI","SP500","TB3MS","UNRATE","USREC",
]

START = "1960-01-01"
END   = "2025-08-01"

In [3]:
# ============================================================
# 1) WIDE dataset (df) depuis Feast
# ============================================================
import pandas as pd

LAG = 6  # ✅ tout lag = 12 (aucune variable contemporaine)

df = load_wide_from_feast(
    "stationary_value:value",
    series_ids,
    start=START,
    end=END
)

# ============================================================
# 2) Convert WIDE -> LONG (obligatoire pour build_unrate_exog_dataset)
# ============================================================
df_stationary = (
    df.reset_index()
      .melt(id_vars="date", var_name="series_id", value_name="value")
)

# ============================================================
# 3) Build target + exog + MLForecast format
# ============================================================
df_model, ts_lr, exog_cols = build_unrate_exog_dataset(df_stationary)

# ============================================================
# 4) ✅ TOUTES les exog laggées de 12 mois (aucune contemporaine)
#    - on crée BUSLOANS_lag12, CPIAUCSL_lag12, ...
#    - on supprime BUSLOANS, CPIAUCSL, ... (contemporaines)
# ============================================================
ts_lr = ts_lr.sort_values(["unique_id", "ds"]).copy()

exog_cols_lag12 = []
for c in exog_cols:
    new_c = f"{c}_lag{LAG}"
    ts_lr[new_c] = ts_lr.groupby("unique_id")[c].shift(LAG)
    exog_cols_lag12.append(new_c)

# supprimer les exog contemporaines
ts_lr = ts_lr.drop(columns=exog_cols)

# mettre à jour la liste exog utilisée par MLForecast
exog_cols = exog_cols_lag12

# drop lignes où les lag12 n'existent pas (12 premiers mois)
ts_lr = ts_lr.dropna(subset=["y"] + exog_cols).reset_index(drop=True)

# ============================================================
# 5) Prints / checks
# ============================================================
print("df (wide) shape:", df.shape)
print("df_stationary (long) shape:", df_stationary.shape)
print("df_model shape:", df_model.shape)
print("ts_lr shape (after lag12):", ts_lr.shape)
print("Exog cols (lag12):", exog_cols)

print("\nPreview:")
print(ts_lr[["unique_id", "ds", "y"] + exog_cols[:5]].head(15))

Using date as the event timestamp. To specify a column explicitly, please name it event_timestamp.
df (wide) shape: (788, 11)
df_stationary (long) shape: (8668, 3)
df_model shape: (788, 12)
ts_lr shape (after lag12): (782, 13)
Exog cols (lag12): ['BUSLOANS_lag6', 'CPIAUCSL_lag6', 'DPCERA3M086SBEA_lag6', 'INDPRO_lag6', 'M2SL_lag6', 'OILPRICEX_lag6', 'RPI_lag6', 'SP500_lag6', 'TB3MS_lag6', 'USREC_lag6']

Preview:
   unique_id         ds    y  BUSLOANS_lag6  CPIAUCSL_lag6  \
0     UNRATE 1960-07-01  0.4       0.011578      -0.006156   
1     UNRATE 1960-08-01  0.4       0.011905      -0.003767   
2     UNRATE 1960-09-01  0.0      -0.008356      -0.005455   
3     UNRATE 1960-10-01  0.4      -0.009098       0.005090   
4     UNRATE 1960-11-01  0.3      -0.000359       0.003383   
5     UNRATE 1960-12-01  1.3       0.014620       0.006777   
6     UNRATE 1961-01-01  1.4      -0.000611      -0.005433   
7     UNRATE 1961-02-01  2.1      -0.016888      -0.004074   
8     UNRATE 1961-03-01  1.

d:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA\3_notebook\ML Experiment\utils.py:171: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  .dt.to_period("M")


# Model settings

In [4]:
# instancier le modèle
models = {"LR": LinearRegression(),
          "RIDGE" : Ridge (), 
          "LGBM" : LGBMRegressor()}

MLF_MODELS = {
    "LR_EXOG_ONLY": lambda freq: MLForecast(
        models=models,
        freq=freq,
        lags=[],               
        date_features=[],      
    )
}

In [5]:
# -----------------------------
# Paramètres
# -----------------------------
H = 12
STEP_SIZE = 1
PI_WINDOWS = 3
LEVELS = [95]
FREQ = "MS"

EXP_START = pd.Timestamp("1990-01-01")
EXP_END   = pd.Timestamp("2025-08-01")   # inclus

In [6]:
def _ensure_ms(x):
    x = pd.Timestamp(x)
    return x.to_period("M").to_timestamp(how="start").normalize()

def _n_windows_monthly(ds_start, ds_end):
    return (ds_end.year - ds_start.year) * 12 + (ds_end.month - ds_start.month) + 1

In [7]:
EXP_START = _ensure_ms(EXP_START)
EXP_END   = _ensure_ms(EXP_END)

# cutoff = ds - h mois
CUTOFF_START = EXP_START - relativedelta(months=H)
CUTOFF_END   = EXP_END   - relativedelta(months=H)
PARTITIONS   = _n_windows_monthly(CUTOFF_START, CUTOFF_END)

print("✅ EXP ds range          :", EXP_START.date(), "→", EXP_END.date())
print("✅ CUTOFF range          :", CUTOFF_START.date(), "→", CUTOFF_END.date())
print("✅ PARTITIONS (n_windows):", PARTITIONS)
print("ts_lr ds range           :", ts_lr["ds"].min().date(), "→", ts_lr["ds"].max().date())

# éviter fuite future (recommandé)
ts_lr = ts_lr[ts_lr["ds"] <= EXP_END].copy()

# -----------------------------
# Instancier le modèle
# -----------------------------
mlf = MLF_MODELS["LR_EXOG_ONLY"](FREQ)

model_names = list(mlf.models.keys())
first_name = next(iter(mlf.models))
print("Running models:", model_names)
print("First model class:", mlf.models[first_name].__class__.__name__)
print("Freq:", FREQ)

✅ EXP ds range          : 1990-01-01 → 2025-08-01
✅ CUTOFF range          : 1989-01-01 → 2024-08-01
✅ PARTITIONS (n_windows): 428
ts_lr ds range           : 1960-07-01 → 2025-08-01
Running models: ['LR', 'RIDGE', 'LGBM']
First model class: LinearRegression
Freq: MS


# Backtesting

In [8]:
# ============================================================
# Backtesting  (style "exercise")
# ============================================================
h = H
step_size = STEP_SIZE
partitions = PARTITIONS
n_windows = PI_WINDOWS
method = "conformal_distribution"
levels = LEVELS

pi = PredictionIntervals(
    h=h,
    n_windows=n_windows,
    method=method,
)

bkt_lr = mlf.cross_validation(
    df=ts_lr,
    h=h,
    step_size=step_size,          # ✅ cutoff mensuel
    n_windows=partitions,         # ✅ nb partitions auto
    prediction_intervals=pi,
    level=levels,
    fitted=True,
    static_features=[],
)

meta = {
    "h": h,
    "step_size": step_size,
    "exp_start": EXP_START,
    "exp_end": EXP_END,
    "cutoff_start": CUTOFF_START,
    "cutoff_end": CUTOFF_END,
    "partitions": partitions,
    "pi_windows": n_windows,
    "conformal_method": method,
    "levels": levels,
}

# -----------------------------
# Filtrer sur l’expérience (sécurité)
# -----------------------------
bkt_lr_eval = bkt_lr[(bkt_lr["ds"] >= EXP_START) & (bkt_lr["ds"] <= EXP_END)].copy()

# ============================================================
# ✅ 1 seule ligne par ds : garder le cutoff le plus récent
# ============================================================
bkt_lr_final = (
    bkt_lr_eval
    .sort_values(["unique_id", "ds", "cutoff"])
    .groupby(["unique_id", "ds"], as_index=False)
    .tail(1)
    .reset_index(drop=True)
)

# -----------------------------
# Checks
# -----------------------------
print("bkt_lr_eval rows         :", len(bkt_lr_eval))
print("bkt_lr_final rows        :", len(bkt_lr_final))
print("bkt_lr_final ds range    :", bkt_lr_final["ds"].min().date(), "→", bkt_lr_final["ds"].max().date())

dup = bkt_lr_final.duplicated(subset=["unique_id", "ds"]).sum()
print("duplicates (unique_id, ds):", dup)

bkt_lr_final.head()

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000164 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 864
[LightGBM] [Info] Number of data points in the train set: 307, number of used features: 10
[LightGBM] [Info] Start training from score 0.071661
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -i

,unique_id,ds,cutoff,y,LR,RIDGE,LGBM,LR-lo-95,LR-hi-95,RIDGE-lo-95,RIDGE-hi-95,LGBM-lo-95,LGBM-hi-95
0,UNRATE,1990-01-01,1989-12-01,0.0,0.293094,-0.284023,-0.006856,-0.075968,0.662156,-0.766392,0.198347,-0.227466,0.213755
1,UNRATE,1990-02-01,1990-01-01,0.1,0.022359,-0.310885,-0.135945,-0.432703,0.477422,-0.854670,0.232899,-0.520775,0.248886
2,UNRATE,1990-03-01,1990-02-01,0.2,0.003990,-0.348619,-0.149455,-0.797507,0.805487,-0.959538,0.262301,-1.503871,1.204960
3,UNRATE,1990-04-01,1990-03-01,0.2,-0.188218,-0.391875,-0.017799,-0.849793,0.473358,-0.984998,0.201247,-0.586013,0.550415
4,UNRATE,1990-05-01,1990-04-01,0.2,-0.144925,-0.397146,-0.116772,-0.715670,0.425821,-0.911522,0.117230,-0.661214,0.427669


# Transformer le Backtesting

In [9]:
bkt_score = bkt_lr_final.copy()
bkt_score["ds"] = pd.to_datetime(bkt_score["ds"], errors="coerce")

# enlever timezone si jamais
if pd.api.types.is_datetime64tz_dtype(bkt_score["ds"]):
    bkt_score["ds"] = bkt_score["ds"].dt.tz_convert(None)

bkt_score = bkt_score.dropna(subset=["ds"])
bkt_score = bkt_score[(bkt_score["ds"] >= EXP_START) & (bkt_score["ds"] <= EXP_END)].reset_index(drop=True)

bins = pd.to_datetime(["1990-01-01","2000-01-01","2009-01-01","2020-01-01","2025-09-01"])
labels = ["1990-1999", "2000-2008", "2009-2019", "2020-end"]

bkt_score["partition"] = pd.cut(bkt_score["ds"], bins=bins, labels=labels, right=False, include_lowest=True)
bkt_score = bkt_score.dropna(subset=["partition"]).reset_index(drop=True)

print(bkt_score[["ds","cutoff","partition"]].head(10))
print(bkt_score["partition"].value_counts().sort_index())

          ds     cutoff  partition
0 1990-01-01 1989-12-01  1990-1999
1 1990-02-01 1990-01-01  1990-1999
2 1990-03-01 1990-02-01  1990-1999
3 1990-04-01 1990-03-01  1990-1999
4 1990-05-01 1990-04-01  1990-1999
5 1990-06-01 1990-05-01  1990-1999
6 1990-07-01 1990-06-01  1990-1999
7 1990-08-01 1990-07-01  1990-1999
8 1990-09-01 1990-08-01  1990-1999
9 1990-10-01 1990-09-01  1990-1999
partition
1990-1999    120
2000-2008    108
2009-2019    132
2020-end      68
Name: count, dtype: int64


C:\Users\Mita\AppData\Local\Temp\ipykernel_15128\1843901458.py:5: DeprecationWarning: is_datetime64tz_dtype is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.DatetimeTZDtype)` instead.
  if pd.api.types.is_datetime64tz_dtype(bkt_score["ds"]):


# Building Leaderbord

In [10]:
tmp = bkt_score.copy()

models = ["LR", "RIDGE", "LGBM"]

# 1) S'assurer que lower <= upper (sécurité)
for m in models:
    lo = f"{m}-lo-95"
    hi = f"{m}-hi-95"
    tmp[[lo, hi]] = np.sort(tmp[[lo, hi]].to_numpy(), axis=1)

# 2) Wide -> Long + features de scoring
rows = []
for m in models:
    lo = f"{m}-lo-95"
    hi = f"{m}-hi-95"

    s = tmp[["unique_id", "ds", "cutoff", "y", "partition"]].copy()
    s["model_label"] = m
    s["model_name"]  = m

    s["forecast"] = tmp[m]
    s["lower"]    = tmp[lo]
    s["upper"]    = tmp[hi]

    s["abs_err"]   = (s["y"] - s["forecast"]).abs()
    s["covered"]   = ((s["y"] >= s["lower"]) & (s["y"] <= s["upper"])).astype(int)
    s["int_width"] = (s["upper"] - s["lower"]).abs()

    rows.append(s)

long_sc = pd.concat(rows, ignore_index=True)

# 3) FIX pandas: partition en string + groupby reset_index
long_sc["partition"] = long_sc["partition"].astype(str)
long_sc = long_sc.loc[:, ~long_sc.columns.duplicated()]

score_df = (
    long_sc
    .groupby(["unique_id", "model_label", "model_name", "partition"], observed=True)
    .agg(
        mae=("abs_err", "mean"),
        coverage=("covered", "mean"),
        width=("int_width", "mean"),
        n=("y", "size"),
    )
    .reset_index()
)

# 4) Top 3 par partition
leaderboard = (
    score_df.sort_values(
        by=["partition", "mae", "coverage", "width"],
        ascending=[True, True, False, True],
    )
    .groupby("partition", as_index=False)
    .head(3)
)

print(score_df.sort_values(["partition", "mae"]).head(20))
print("\nTop 3 par partition:")
print(leaderboard)

   unique_id model_label model_name  partition       mae  coverage     width  \
0     UNRATE        LGBM       LGBM  1990-1999  0.412094  0.791667  1.450651   
4     UNRATE          LR         LR  1990-1999  0.421220  0.800000  1.419842   
8     UNRATE       RIDGE      RIDGE  1990-1999  0.423586  0.800000  1.452102   
1     UNRATE        LGBM       LGBM  2000-2008  0.358597  0.703704  1.109895   
5     UNRATE          LR         LR  2000-2008  0.377789  0.648148  0.981396   
9     UNRATE       RIDGE      RIDGE  2000-2008  0.389928  0.685185  1.018258   
2     UNRATE        LGBM       LGBM  2009-2019  0.503702  0.840909  1.951557   
10    UNRATE       RIDGE      RIDGE  2009-2019  0.546811  0.795455  1.958084   
6     UNRATE          LR         LR  2009-2019  0.597483  0.780303  1.902029   
3     UNRATE        LGBM       LGBM   2020-end  1.725460  0.720588  5.973706   
7     UNRATE          LR         LR   2020-end  1.776643  0.764706  6.250181   
11    UNRATE       RIDGE      RIDGE   20

# MLFLOW

In [11]:
import mlflow

# =====================================================
# 0) (Optionnel mais recommandé) Tracking URI
# =====================================================
mlflow.set_tracking_uri("http://127.0.0.1:5000")

# =====================================================
# 1) Set experiment
# =====================================================
experiment_name = "Multivariate_experiment_design"
mlflow.set_experiment(experiment_name)

# =====================================================
# 2) DataFrame à logger
# =====================================================
df_log = score_df.copy()  # ou leaderboard_global

# =====================================================
# 3) Logging loop
# =====================================================
for idx, row in df_log.iterrows():
    
    partition = row["partition"] if "partition" in df_log.columns else "global"
    run_name = f"{row['model_label']}_partition_{partition}"
    
    with mlflow.start_run(run_name=run_name):
        
        # ======================
        # PARAMETERS
        # ======================
        mlflow.log_param("model_label", row["model_label"])
        mlflow.log_param("model_name", row["model_name"])
        mlflow.log_param("partition", partition)
        mlflow.log_param("horizon", 12)
        mlflow.log_param("lags", list(range(1, 25)))
        mlflow.log_param("date_features", ["year", "month", "quarter"])
        mlflow.log_param("conformal_method", "conformal_distribution")
        
        # ======================
        # METRICS
        # ======================
        mlflow.log_metric("mae", float(row["mae"]))
        mlflow.log_metric("coverage", float(row["coverage"]))
        mlflow.log_metric("width", float(row["width"]))
        
        # ======================
        # TAGS (optionnel mais pro)
        # ======================
        mlflow.set_tag("project", "UNRATE_forecasting")
        mlflow.set_tag("framework", "MLForecast")
        mlflow.set_tag("model_family", row["model_name"])

print("Logging terminé.")

🏃 View run LGBM_partition_1990-1999 at: http://127.0.0.1:5000/#/experiments/7/runs/a5eee73d9c4144918835efbb23c31ef9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7
🏃 View run LGBM_partition_2000-2008 at: http://127.0.0.1:5000/#/experiments/7/runs/f410a14d0d5d442fa6d7909c42809444
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7
🏃 View run LGBM_partition_2009-2019 at: http://127.0.0.1:5000/#/experiments/7/runs/2f14e3abc89e4216a3b1bc05a2a06289
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7
🏃 View run LGBM_partition_2020-end at: http://127.0.0.1:5000/#/experiments/7/runs/e8a3129cc160420f89c4cbfb6198ff89
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7
🏃 View run LR_partition_1990-1999 at: http://127.0.0.1:5000/#/experiments/7/runs/68f8232f28d24e369c4fc016e5d411a4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7
🏃 View run LR_partition_2000-2008 at: http://127.0.0.1:5000/#/experiments/7/runs/e4cf3a9d287344f6a1acba45178e0cae
🧪 View ex

# Vérif

In [12]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error

# =========================================================
# MAE pour LR (global + par partition) à partir de bkt_lr_final
# Colonnes attendues: unique_id, ds, y, LR
# =========================================================

EXP_START = pd.Timestamp("1990-01-01")
EXP_END   = pd.Timestamp("2025-08-01")

df = bkt_lr_final.copy()

# ds propre
df["ds"] = pd.to_datetime(df["ds"], errors="coerce")
if pd.api.types.is_datetime64tz_dtype(df["ds"]):
    df["ds"] = df["ds"].dt.tz_convert(None)

# garder lignes valides
df = df.dropna(subset=["ds", "y", "LR"]).copy()
df = df[(df["ds"] >= EXP_START) & (df["ds"] <= EXP_END)].reset_index(drop=True)

# -----------------------------
# MAE global
# -----------------------------
mae_all = mean_absolute_error(df["y"], df["LR"])
print("✅ MAE moyen (LR) — GLOBAL :", float(mae_all))

# -----------------------------
# MAE par partition macro (based on ds)
# -----------------------------
bins = pd.to_datetime(["1990-01-01","2000-01-01","2009-01-01","2020-01-01","2025-09-01"])
labels = ["1990-1999", "2000-2008", "2009-2019", "2020-fin"]

df["partition"] = pd.cut(df["ds"], bins=bins, labels=labels, right=False, include_lowest=True)
df = df.dropna(subset=["partition"]).copy()

mae_by_partition = (
    df.assign(abs_err=np.abs(df["y"] - df["LR"]))
      .groupby("partition")["abs_err"]
      .mean()
      .sort_index()
)

n_by_partition = df.groupby("partition").size().sort_index()

print("\n✅ MAE moyen (LR) — PAR PARTITION :")
print(mae_by_partition)

print("\n✅ N obs — PAR PARTITION :")
print(n_by_partition)

# (optionnel) MAE pondéré (≈ MAE global)
mae_weighted = float((mae_by_partition * n_by_partition).sum() / n_by_partition.sum())
print("\n✅ MAE moyen (LR) — pondéré partitions :", mae_weighted)

✅ MAE moyen (LR) — GLOBAL : 0.6799697179008874

✅ MAE moyen (LR) — PAR PARTITION :
partition
1990-1999    0.421220
2000-2008    0.377789
2009-2019    0.597483
2020-fin     1.776643
Name: abs_err, dtype: float64

✅ N obs — PAR PARTITION :
partition
1990-1999    120
2000-2008    108
2009-2019    132
2020-fin      68
dtype: int64

✅ MAE moyen (LR) — pondéré partitions : 0.6799697179008874


C:\Users\Mita\AppData\Local\Temp\ipykernel_15128\3500170289.py:17: DeprecationWarning: is_datetime64tz_dtype is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.DatetimeTZDtype)` instead.
  if pd.api.types.is_datetime64tz_dtype(df["ds"]):
C:\Users\Mita\AppData\Local\Temp\ipykernel_15128\3500170289.py:41: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("partition")["abs_err"]
C:\Users\Mita\AppData\Local\Temp\ipykernel_15128\3500170289.py:46: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  n_by_partition = df.groupby("partition").size().sort_index()
